In [47]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

True

In [48]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)


# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [49]:
# query_com = """
# SELECT DISTINCT Pedido,
#     c.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2026-09-19'
#     AND v.Fecha <= '2026-09-19'
#     AND Canal IN ('Shopify')
#     AND v.Venta_Neta > 0
#     AND V.Cliente NOT IN ('C22222222222')
# GROUP BY  Pedido,
#     c.Cliente,
#     c.Documento,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_com, engine)

# df_ventas.head()

In [50]:
# CONSULTAS CUMPLEAÑOS

query_data = """
SELECT DISTINCT Pedido,
    v.Cliente, 
	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
	cc.Email, cc.Celular, 
	v.Fecha AS Fecha_Venta,
	Canal,
    SUM(Venta_Neta) AS Venta 
FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'SEPTIEMBRE_2026')
	AND Fecha BETWEEN '2026-09-30' AND '2026-09-30'
    AND Canal = 'Shopify'
    AND V.Cliente NOT IN ('C22222222222')
GROUP BY  Pedido,
    v.Cliente,
    cc.Nombres + ' ' +  cc.Apellidos,
    cc.Email,
    cc.Celular,
    v.Fecha,
    v.Canal
"""


# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_data, engine)
df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,104118,C1018419250,ZARETH GALVAN,zcgalvan@gmail.com,3243670638,2026-09-30,Shopify,190958.17
1,104138,C52778959,LUISA GOMEZ,luisa279@gmail.com,3158319404,2026-09-30,Shopify,189067.50
2,104140,C55300717,MARIA VICTORIA CASTAÑO LEMUS,ma_victoriacl@hotmail.com,3212542353,2026-09-30,Shopify,562368.10
3,104145,C72000371,JAIRO MONTOYA,jrmontoy@yahoo.com,3202496869,2026-09-30,Shopify,258286.95
4,104147,C43628359,PAULA ANDREA ECHEVERRI,paecheverri@hotmail.com,3183821299,2026-09-30,Shopify,400823.09


In [51]:
# Leer archivo
df_revie = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_revie[['Email', 'First Name', 'Last Name', 'Phone', 'Company']].copy()

# 4. Normalizar correos
df_campania['Email'] = df_campania['Email'].str.lower().str.strip()


In [52]:
# Normalizar df_ventas
df_ventas['Email'] = df_ventas['Email'].str.lower().str.strip()
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_atribucion = pd.merge(df_ventas, df_campania, on='Email', how='inner')

# Sumar la venta total atribuida
total_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {total_venta:,.2f}"  )
# Resultado final
df_atribucion.head()

Total de Ventas: 0
Total de venta atribuida: 0.00


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,First Name,Last Name,Phone,Company


In [53]:
df_atribucion.to_excel('Ventas.xlsx', index=False)